# GeoPhys in five minutes

GeoPhys passes each frame of a video through a frozen image encoder and looks at the
*trajectory* the frame features trace out. Physically implausible motion bends and breaks
that trajectory. Nothing is trained.

This notebook takes six pairs from the [LikePhys](https://github.com/yuanjianhao508/likephys)
benchmark, each a plausible video and an implausible version of the same scene, and shows
(1) the videos, (2) the trajectory geometry frame by frame, (3) one GeoPhys score per video
and (4), optionally, the full method. The pairs were chosen to show clear cases, one per type
of violation; the accuracy on the whole benchmark is what `python -m geophys.detect`
reproduces (see the README).

A GPU runtime is recommended (Runtime > Change runtime type).

In [ ]:
# install (in Colab); skip if you already have GeoPhys
!git clone -q https://github.com/ChristianInterno/GeoPhys.git
!pip install -q --no-deps -e GeoPhys   # Colab already has torch, transformers, opencv
import sys; sys.path.insert(0, "GeoPhys")   # importable without restarting the runtime

## 1. Six LikePhys pairs

Each folder holds `plausible.mp4` and `implausible.mp4`. The violations: frames out of
temporal order, a flag that shatters, a collision driven by an invisible force, a ball that
changes size as it falls, cloth passing through an object, and a block that teleports.

In [ ]:
import os, glob, base64
import numpy as np, matplotlib.pyplot as plt
from IPython.display import HTML, display
from geophys.backbones import load_frames

EX = next(p for p in ("GeoPhys/notebooks/examples/likephys", "examples/likephys",
                      "notebooks/examples/likephys") if os.path.isdir(p))
cases = sorted(os.listdir(EX))
pairs = {c: {n: glob.glob(os.path.join(EX, c, n + ".*"))[0] for n in ("plausible", "implausible")}
         for c in cases}
print("\n".join(cases))

def show_videos(case):
    tags = "".join(
        '<figure style="display:inline-block;margin:4px"><video width="280" controls loop muted autoplay '
        'src="data:video/mp4;base64,%s"></video><figcaption>%s</figcaption></figure>'
        % (base64.b64encode(open(pairs[case][n], "rb").read()).decode(), n) for n in ("plausible", "implausible"))
    display(HTML("<b>%s</b><br>%s" % (case.replace("_", " "), tags)))

def show_frames(case, k=5):
    fig, ax = plt.subplots(2, k, figsize=(2.2 * k, 4.6))
    for i, n in enumerate(("plausible", "implausible")):
        fr = load_frames(pairs[case][n])
        for j, t in enumerate(np.linspace(0, len(fr) - 1, k).astype(int)):
            ax[i, j].imshow(fr[t]); ax[i, j].axis("off")
            ax[i, j].set_title("%s, frame %d" % (n, t), fontsize=8)
    fig.suptitle(case.replace("_", " ")); plt.tight_layout(); plt.show()

for c in cases:
    show_videos(c)      # if your browser does not play them, the frames below show the same
    show_frames(c)

## 2. The trajectory, frame by frame

Each frame becomes one feature vector *z*<sub>t</sub> (DINOv2, block 12, patch tokens
averaged), and the video becomes the trajectory *z*<sub>1</sub>, *z*<sub>2</sub>, ...
Four quantities along it:

- **speed**: how far the trajectory moves between two frames, |*z*<sub>t+1</sub> - *z*<sub>t</sub>|
- **curvature**: the angle between the step into a frame and the step out of it
- **acceleration**: how much consecutive steps differ, |*z*<sub>t+1</sub> - 2*z*<sub>t</sub> + *z*<sub>t-1</sub>|
- **prediction residual**: the acceleration relative to the speed, i.e. how far the next
  frame lands from where a straight continuation would put it

Plausible motion gives low, steady curves (blue); a violation shows up as spikes and
jumps (red).

In [ ]:
import torch
from geophys.backbones import get_backbone, prep_frames
from geophys.extract import _forward

device = "cuda" if torch.cuda.is_available() else "cpu"
enc = get_backbone("dinov2", device)

def trajectory(path, layer="12"):
    pooled, _ = _forward(enc, prep_frames(load_frames(path), "direct"), device)
    return pooled[layer]["mean"].double().cpu().numpy()

def frame_signals(z):
    v = np.diff(z, axis=0)                                    # steps between frames
    s = np.linalg.norm(v, axis=1)                             # speed
    a = np.linalg.norm(np.diff(v, axis=0), axis=1)            # acceleration
    cos = (v[:-1] * v[1:]).sum(1) / (s[:-1] * s[1:] + 1e-12)
    return {"speed": s,
            "curvature (degrees)": np.degrees(np.arccos(np.clip(cos, -1, 1))),
            "acceleration": a,
            "prediction residual": a / (s[:-1] + 1e-12)}

Z = {c: {n: trajectory(pairs[c][n]) for n in pairs[c]} for c in cases}

keys = list(frame_signals(np.random.rand(4, 2)))
fig, ax = plt.subplots(len(cases), 4, figsize=(16, 2.4 * len(cases)), squeeze=False)
for i, c in enumerate(cases):
    for n, col in (("plausible", "tab:blue"), ("implausible", "tab:red")):
        sig = frame_signals(Z[c][n])
        for j, k in enumerate(keys):
            ax[i, j].plot(sig[k], color=col, label=n)
    ax[i, 0].set_ylabel(c.replace("_", " "), fontsize=9)
    ax[i, 3].set_yscale("log")
for j, k in enumerate(keys):
    ax[0, j].set_title(k)
    ax[-1, j].set_xlabel("frame")
ax[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3. One score per video

`geophys.rate` reads five signals off each trajectory (curvature, angle consistency, speed
variation, acceleration, prediction residual), turns each into a percentile among the videos
scored together, and averages them: 0.5 is typical, higher is less plausible. The default
encoder is DINOv3; DINOv2 is used here because it needs no licence step on Hugging Face.

In [ ]:
import geophys, pandas as pd

paths = [pairs[c][n] for c in cases for n in ("plausible", "implausible")]
score = {r["video"]: r["score"] for r in geophys.rate(paths, backbone="dinov2", device=device)}
table = pd.DataFrame([{"pair": c,
                       "plausible": score[os.path.abspath(pairs[c]["plausible"])],
                       "implausible": score[os.path.abspath(pairs[c]["implausible"])]} for c in cases])
table["implausible scores higher"] = table["implausible"] > table["plausible"]

table.set_index("pair")[["plausible", "implausible"]].plot.barh(
    color=["tab:blue", "tab:red"], figsize=(8, 4), xlabel="GeoPhys score (higher = less plausible)")
plt.axvline(0.5, color="gray", lw=0.8, ls="--"); plt.tight_layout(); plt.show()
table.round(3)

## 4. The full method (optional)

`geophys.compare(a, b)` says which of two videos violates physics with the paper's GeoPhys+
configuration, which combines four encoders (DINOv2, DINOv3, CORnet-S, VOneNet).
`geophys.pick([...])` returns the most plausible of several candidates in the same way.

To run it: accept the DINOv3 licence on its
[Hugging Face page](https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m),
set `RUN_FULL = True` and run the cell (it installs CORnet-S and VOneNet and asks for a
Hugging Face token). Feature extraction takes a few minutes.

In [ ]:
RUN_FULL = False

if RUN_FULL:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/dicarlolab/CORnet", "git+https://github.com/dicarlolab/vonenet"], check=True)
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
if RUN_FULL:
    verdicts = geophys.compare([(pairs[c]["plausible"], pairs[c]["implausible"]) for c in cases], device=device)
    for c, v in zip(cases, verdicts):
        print("%-36s judged implausible: %-16s confidence %.2f"
              % (c, os.path.basename(v["implausible"]), v["confidence"]))